# ECHO Training Notebook
Trains Qwen2.5-7B to predict its own correctness using GRPO + OpenEnv

In [ ]:
# Install dependencies
!pip install -q "trl>=0.8.0" "peft" "transformers" "datasets" "huggingface_hub"
!pip install -q "openenv-core[core]>=0.2.0" || pip install -q git+https://github.com/meta-pytorch/OpenEnv.git
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import os
import requests
import json
import numpy as np
from huggingface_hub import login

# Authenticate
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # Set in Colab secrets
if HF_TOKEN:
    login(HF_TOKEN)

# Connect to live ECHO environment on HuggingFace Spaces
ECHO_SPACE_URL = "https://vikaspandey582003-echo-ultimate.hf.space"

# Test connection
resp = requests.get(f"{ECHO_SPACE_URL}/health", timeout=10)
print(f"Space status: {resp.json()}")

In [ ]:
# Simple HTTP client for the ECHO environment
class EchoEnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")
    
    def reset(self):
        r = requests.post(f"{self.base_url}/reset", timeout=30)
        r.raise_for_status()
        return r.json()
    
    def step(self, response_text: str):
        # OpenEnv servers may accept either {"response": ...} or {"action": {"response": ...}}
        payloads = [
            {"response": response_text},
            {"action": {"response": response_text}},
        ]
        last_error = None
        for payload in payloads:
            try:
                r = requests.post(f"{self.base_url}/step", json=payload, timeout=30)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last_error = e
        raise RuntimeError(f"Step request failed for all payload formats: {last_error}")
    
    def get_metrics(self):
        r = requests.get(f"{self.base_url}/metrics", timeout=10)
        r.raise_for_status()
        return r.json()

env = EchoEnvClient(ECHO_SPACE_URL)

# Test: reset and take a step
obs = env.reset()
print("Question:", obs.get("question", ""))
result = env.step("<confidence>70</confidence><answer>test answer</answer>")
print("Step response keys:", list(result.keys()))

In [ ]:
# Load model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

SYSTEM_PROMPT = """You are a calibrated AI assistant. For every question:
1. Think step-by-step (optional: use <think>...</think> tags)
2. Output your confidence as an integer 0-100: <confidence>INTEGER</confidence>
3. Output your answer: <answer>YOUR ANSWER</answer>

Be honest about uncertainty. Overconfidence is penalized heavily.

CRITICAL: You MUST use <confidence> and <answer> tags. Responses without these tags score -0.5 reward automatically. Example of correct format:
<think>The capital of France is Paris, I am very sure.</think>
<confidence>95</confidence>
<answer>Paris</answer>"""

# Build dataset from ECHO environment
def build_training_dataset(n_samples=500):
    samples = []
    for _ in range(n_samples):
        obs = env.reset()
        question = obs.get("question", "")
        samples.append({
            "prompt": f"{SYSTEM_PROMPT}\n\nQuestion: {question}",
            "question": question,
        })
    return Dataset.from_list(samples)

print("Building training dataset from live environment...")
dataset = build_training_dataset(500)
print(f"Dataset size: {len(dataset)}")

In [ ]:
# GRPO reward function — calls live OpenEnv environment
ece_history = []
reward_history = []
confidence_eval_history = []
outcome_history = []

def _extract_step_values(result: dict):
    # Supports both flat and OpenEnv-shaped responses.
    obs = result.get("observation") or result.get("obs") or result.get("state") or {}
    info = result.get("info") or {}

    reward = result.get("reward", info.get("reward", obs.get("reward", 0.0)))
    ece = result.get("ece", info.get("ece", obs.get("ece", 0.5)))
    conf = result.get("confidence", obs.get("confidence", None))
    is_correct = result.get("is_correct", obs.get("is_correct", info.get("was_correct", None)))

    return float(reward), float(ece), conf, is_correct

def echo_reward_function(completions, prompts=None, **kwargs):
    """
    Reward function that evaluates each completion against the live ECHO environment.
    This is the core of GRPO training — the environment provides the reward signal.
    """
    rewards = []
    for i, completion in enumerate(completions):
        try:
            # Reset for each completion so reward is grounded to a fresh environment question.
            env.reset()

            # Each completion is evaluated by the running OpenEnv Space.
            result = env.step(completion)
            reward, ece, conf, is_correct = _extract_step_values(result)

            ece_history.append(ece)
            reward_history.append(reward)
            if conf is not None:
                confidence_eval_history.append(float(conf) / 100.0)
            if is_correct is not None:
                outcome_history.append(1.0 if bool(is_correct) else 0.0)
            rewards.append(reward)

        except Exception as e:
            print(f"Env step failed: {e}")
            rewards.append(-0.5)  # penalty for failed step

    return rewards

# Alias used by the sanity check cell below
echo_reward = echo_reward_function

In [ ]:
# ============================================================
# PRE-TRAINING SANITY CHECK — run this before starting training
# If all 3 checks pass, training will work. If any fail, fix first.
# ============================================================
print("=== PRE-TRAINING SANITY CHECK ===\n")

# 1. Test environment connection
obs = env.reset()
assert "question" in obs, "❌ /reset broken — check Space is running"
print(f"✅ Environment connected: {obs['question'][:70]}...")

# 2. Test reward function with a known good response
good_response = "<think>Let me think carefully about this.</think><confidence>75</confidence><answer>Paris</answer>"
result = env.step(good_response)
assert "reward" in result or "state" in result, "❌ /step broken — check Space is running"
reward_val = result.get("reward", result.get("state", {}).get("reward", "?"))
ece_val    = result.get("ece",    result.get("state", {}).get("ece",    "?"))
print(f"✅ /step working: reward={reward_val}, ece={ece_val}")

# 3. Test reward function returns sensible values
test_responses = [
    "<confidence>80</confidence><answer>42</answer>",                      # good format
    "<think>hmm</think><confidence>60</confidence><answer>Paris</answer>", # good format with think
    "I think the answer is Paris, I am sure about this.",                   # BAD format — no tags
]
rewards = echo_reward(test_responses)
print(f"✅ Reward function outputs: {[round(r, 3) for r in rewards]}")
print(f"   good_format_1={rewards[0]:.3f}  good_format_2={rewards[1]:.3f}  bad_format={rewards[2]:.3f}")

assert rewards[2] < max(rewards[0], rewards[1]), (
    f"❌ Bad format not being penalized! rewards={rewards}. "
    "Check echo_reward_function — parser may not be filtering correctly."
)

print()
print("=" * 50)
print("✅ ALL CHECKS PASSED — safe to start training!")
print(f"   Good format reward: {rewards[0]:.3f}")
print(f"   Bad  format reward: {rewards[2]:.3f}")
print(f"   Penalty gap: {rewards[0] - rewards[2]:.3f}")
print()
print("⚠️  WATCH for these in first 30 training steps:")
print("   GOOD: rewards between -0.5 and +0.8 (mixed)")
print("   BAD : all rewards exactly -0.5 → stop & report")
print("=" * 50)

In [ ]:
# Configure GRPO training — OPTIMIZED for A10G small (~2.5 hrs, ~$3-4 cost)
# Hardware: A10G small ($1.05/hr) — 3x faster than T4 for 7B models
# max_completion_length=256: enough for reasoning, 2x faster than 512

# Rebuild dataset for A10G run
dataset_a10g = build_training_dataset(300)
print(f"Dataset: {len(dataset_a10g)} samples")

training_args = GRPOConfig(
    output_dir="echo_grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,    # effective batch = 8, keep for GRPO stability
    learning_rate=2e-5,
    warmup_steps=20,
    logging_steps=5,
    save_steps=50,
    bf16=True,                        # A10G supports bfloat16 — better than fp16
    fp16=False,
    report_to="none",
    max_completion_length=256,        # 256 = enough reasoning space, 2x faster than 512
    num_generations=4,                # GRPO group size — do NOT reduce
    temperature=0.8,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[echo_reward_function],
    train_dataset=dataset_a10g,
    tokenizer=tokenizer,
)

print("=" * 55)
print("🚀  ECHO GRPO Training — A10G small + 256 tokens")
print("    300 samples | 1 epoch | grad_accum=8")
print("    Estimated: ~2.5 hrs | Cost: ~$3-4")
print("=" * 55)
print()
print("Watch step output — after step 5 you should see:")
print("  GOOD: rewards mixed between -0.5 and +0.8")
print("  BAD : all rewards exactly -0.5 → stop & report")
print()
trainer.train()
print("\n✅ Training complete!")

In [ ]:
# Plot ECE curve, reward curve, and reliability diagram
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

# ECE curve
if ece_history:
    window = 50
    smoothed = [np.mean(ece_history[max(0, i - window):i + 1]) for i in range(len(ece_history))]
    ax1.plot(ece_history, alpha=0.3, color='blue', label='Raw ECE')
    ax1.plot(smoothed, color='blue', linewidth=2, label='Smoothed ECE')
    ax1.axhline(y=0.15, color='green', linestyle='--', label='Good threshold (0.15)')
    ax1.axhline(y=0.20, color='orange', linestyle='--', label='Acceptable (0.20)')
    ax1.set_xlabel('Training Steps')
    ax1.set_ylabel('ECE (lower = better)')
    ax1.set_title('ECHO: ECE During GRPO Training')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# Reward curve
if reward_history:
    window = 50
    smoothed_r = [np.mean(reward_history[max(0, i - window):i + 1]) for i in range(len(reward_history))]
    ax2.plot(reward_history, alpha=0.3, color='green', label='Raw Reward')
    ax2.plot(smoothed_r, color='green', linewidth=2, label='Smoothed Reward')
    ax2.set_xlabel('Training Steps')
    ax2.set_ylabel('Reward')
    ax2.set_title('ECHO: Reward During GRPO Training')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# Reliability diagram
if confidence_eval_history and outcome_history and len(confidence_eval_history) == len(outcome_history):
    n_bins = 10
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    accs = []
    confs = []

    conf_arr = np.array(confidence_eval_history)
    out_arr = np.array(outcome_history)

    for i in range(n_bins):
        mask = (conf_arr >= bins[i]) & (conf_arr < bins[i + 1])
        if i == n_bins - 1:
            mask = (conf_arr >= bins[i]) & (conf_arr <= bins[i + 1])
        if np.any(mask):
            accs.append(float(np.mean(out_arr[mask])))
            confs.append(float(np.mean(conf_arr[mask])))
        else:
            accs.append(np.nan)
            confs.append(np.nan)

    ax3.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
    ax3.plot(bin_centers, accs, marker='o', linewidth=2, color='purple', label='Model')
    ax3.set_xlabel('Predicted confidence')
    ax3.set_ylabel('Empirical accuracy')
    ax3.set_title('Reliability Diagram')
    ax3.set_xlim(0, 1)
    ax3.set_ylim(0, 1)
    ax3.grid(True, alpha=0.3)
    ax3.legend()

plt.tight_layout()
plt.savefig("echo_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Final ECE: {ece_history[-1]:.4f}" if ece_history else "No ECE data")

In [ ]:
# Save and push adapter to HF Hub
model.save_pretrained("echo_lora_adapter")
tokenizer.save_pretrained("echo_lora_adapter")

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="echo_lora_adapter",
    repo_id="Vikaspandey582003/echo-calibration-adapter",
    repo_type="model",
    commit_message="ECHO GRPO-trained calibration adapter - Hackathon submission",
)
print("Adapter pushed to HF Hub!")
print("Model: https://huggingface.co/Vikaspandey582003/echo-calibration-adapter")

# Baseline vs Trained Evaluation
Run this section to get the before/after comparison judges need to see.

**Step 1:** Run cells below with `EVAL_MODE = "baseline"` → saves `baseline_results.json`  
**Step 2:** After training finishes, run again with `EVAL_MODE = "trained"` → saves `trained_results.json`  
**Step 3:** Run the comparison cell to generate the plot

In [ ]:

# ============================================================
# BASELINE EVALUATION — run BEFORE loading trained adapter
# Set EVAL_MODE = "trained" after training to compare
# ============================================================
import torch, json, re, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

EVAL_MODE = "baseline"   # change to "trained" after training finishes
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "Vikaspandey582003/echo-calibration-adapter"
N_EVAL = 50   # number of test questions

TEST_QUESTIONS = [
    {"question": "What is the capital of France?",         "answer": "Paris",       "domain": "factual"},
    {"question": "What is 17 × 24?",                       "answer": "408",         "domain": "math"},
    {"question": "Who wrote Romeo and Juliet?",            "answer": "Shakespeare", "domain": "factual"},
    {"question": "What is the square root of 144?",        "answer": "12",          "domain": "math"},
    {"question": "What planet is closest to the Sun?",     "answer": "Mercury",     "domain": "factual"},
    {"question": "What is 2^10?",                          "answer": "1024",        "domain": "math"},
    {"question": "What is the chemical symbol for gold?",  "answer": "Au",          "domain": "science"},
    {"question": "Who invented the telephone?",            "answer": "Bell",        "domain": "factual"},
    {"question": "What is the speed of light in km/s?",   "answer": "300000",      "domain": "science"},
    {"question": "What is the largest planet?",            "answer": "Jupiter",     "domain": "factual"},
    {"question": "What is 15% of 200?",                   "answer": "30",          "domain": "math"},
    {"question": "What is the boiling point of water in Celsius?", "answer": "100", "domain": "science"},
    {"question": "Who painted the Mona Lisa?",             "answer": "da Vinci",    "domain": "factual"},
    {"question": "What is the atomic number of Carbon?",   "answer": "6",           "domain": "science"},
    {"question": "What is 3! (3 factorial)?",              "answer": "6",           "domain": "math"},
    {"question": "What continent is Egypt in?",            "answer": "Africa",      "domain": "factual"},
    {"question": "What gas do plants absorb from the air?","answer": "CO2",         "domain": "science"},
    {"question": "What is the hardest natural substance?", "answer": "diamond",     "domain": "science"},
    {"question": "What is 1000 divided by 8?",             "answer": "125",         "domain": "math"},
    {"question": "Who was the first US President?",        "answer": "Washington",  "domain": "factual"},
]

SYSTEM_PROMPT = """You are a knowledgeable assistant that answers questions and expresses calibrated confidence.

Always respond in this exact format:
<think>Brief reasoning about the question.</think>
<confidence>NUMBER</confidence>
<answer>Your answer here</answer>

Where NUMBER is your confidence percentage (0-100). Be honest: 50 means you're guessing, 95 means you're very sure."""

def load_model(mode):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
    )
    if mode == "trained":
        print(f"Loading LoRA adapter from {ADAPTER_REPO}...")
        model = PeftModel.from_pretrained(base, ADAPTER_REPO)
        model = model.merge_and_unload()
        print("Adapter merged.")
    else:
        model = base
    model.eval()
    return model, tokenizer

def parse_response(text):
    conf = re.search(r'<confidence>(\d+)</confidence>', text)
    ans  = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    return {
        "confidence": int(conf.group(1)) if conf else 50,
        "answer": ans.group(1).strip().lower() if ans else "",
    }

def is_correct(pred_answer, true_answer):
    pred = pred_answer.lower().strip()
    true = true_answer.lower().strip()
    return true in pred or pred in true

def run_evaluation(model, tokenizer, questions):
    results = []
    for i, q in enumerate(questions):
        prompt = f"{SYSTEM_PROMPT}\n\nQuestion: {q['question']}"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        parsed = parse_response(response)
        correct = is_correct(parsed["answer"], q["answer"])
        results.append({
            "question": q["question"],
            "true_answer": q["answer"],
            "predicted_answer": parsed["answer"],
            "confidence": parsed["confidence"],
            "correct": correct,
            "response": response[:200],
        })
        print(f"[{i+1}/{len(questions)}] Conf={parsed['confidence']}% Correct={correct} | Q: {q['question'][:40]}")
    return results

def compute_metrics(results):
    confs = np.array([r["confidence"] / 100.0 for r in results])
    correct = np.array([1.0 if r["correct"] else 0.0 for r in results])
    accuracy = correct.mean()
    avg_confidence = confs.mean()
    # ECE — Expected Calibration Error
    bins = np.linspace(0, 1, 11)
    ece = 0.0
    for i in range(len(bins)-1):
        mask = (confs >= bins[i]) & (confs < bins[i+1])
        if mask.sum() > 0:
            bin_acc = correct[mask].mean()
            bin_conf = confs[mask].mean()
            ece += mask.sum() / len(confs) * abs(bin_acc - bin_conf)
    overconfidence = ((confs > 0.7) & (correct == 0)).mean()
    return {
        "accuracy": float(accuracy),
        "avg_confidence": float(avg_confidence),
        "ece": float(ece),
        "overconfidence_rate": float(overconfidence),
        "n": len(results),
    }

print(f"Running {EVAL_MODE} evaluation on {len(TEST_QUESTIONS)} questions...")
model, tokenizer = load_model(EVAL_MODE)
results = run_evaluation(model, tokenizer, TEST_QUESTIONS)
metrics = compute_metrics(results)
print(f"\n{'='*50}")
print(f"RESULTS ({EVAL_MODE.upper()})")
print(f"{'='*50}")
print(f"Accuracy:           {metrics['accuracy']*100:.1f}%")
print(f"Avg Confidence:     {metrics['avg_confidence']*100:.1f}%")
print(f"ECE (lower=better): {metrics['ece']:.4f}")
print(f"Overconfidence:     {metrics['overconfidence_rate']*100:.1f}%")

with open(f"{EVAL_MODE}_results.json", "w") as f:
    json.dump({"metrics": metrics, "results": results}, f, indent=2)
print(f"\nSaved to {EVAL_MODE}_results.json")


In [ ]:

# ============================================================
# COMPARISON PLOT — run after BOTH baseline + trained are done
# Generates the plot judges need to see in your README
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, json

with open("baseline_results.json") as f:
    baseline = json.load(f)
with open("trained_results.json") as f:
    trained = json.load(f)

bm = baseline["metrics"]
tm = trained["metrics"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("ECHO: Baseline vs GRPO-Trained Model", fontsize=14, fontweight='bold')

# Bar chart 1: Key metrics
metrics_labels = ["Accuracy", "ECE\n(lower=better)", "Overconfidence\nRate"]
baseline_vals = [bm["accuracy"], bm["ece"], bm["overconfidence_rate"]]
trained_vals  = [tm["accuracy"], tm["ece"], tm["overconfidence_rate"]]
x = np.arange(len(metrics_labels))
w = 0.35
axes[0].bar(x - w/2, baseline_vals, w, label="Base Model", color="#FF6B6B", alpha=0.85)
axes[0].bar(x + w/2, trained_vals,  w, label="GRPO Trained", color="#4ECDC4", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_labels)
axes[0].set_ylabel("Score")
axes[0].set_title("Key Metrics Comparison")
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for i, (b, t) in enumerate(zip(baseline_vals, trained_vals)):
    axes[0].text(i - w/2, b + 0.005, f"{b:.3f}", ha='center', va='bottom', fontsize=8)
    axes[0].text(i + w/2, t + 0.005, f"{t:.3f}", ha='center', va='bottom', fontsize=8, color='teal')

# Reliability diagram (calibration curve)
def reliability_curve(results):
    confs = np.array([r["confidence"] / 100.0 for r in results])
    correct = np.array([1.0 if r["correct"] else 0.0 for r in results])
    bins = np.linspace(0, 1, 6)
    centers, accs = [], []
    for i in range(len(bins)-1):
        mask = (confs >= bins[i]) & (confs < bins[i+1])
        if mask.sum() > 0:
            centers.append(confs[mask].mean())
            accs.append(correct[mask].mean())
    return centers, accs

bc, ba = reliability_curve(baseline["results"])
tc, ta = reliability_curve(trained["results"])
axes[1].plot([0,1],[0,1], '--', color='gray', label='Perfect calibration', linewidth=2)
axes[1].plot(bc, ba, 'o-', color='#FF6B6B', label='Base Model', linewidth=2, markersize=8)
axes[1].plot(tc, ta, 's-', color='#4ECDC4', label='GRPO Trained', linewidth=2, markersize=8)
axes[1].set_xlabel("Predicted Confidence")
axes[1].set_ylabel("Empirical Accuracy")
axes[1].set_title("Reliability Diagram\n(closer to diagonal = better calibrated)")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)

# Confidence distribution
b_confs = [r["confidence"] for r in baseline["results"]]
t_confs = [r["confidence"] for r in trained["results"]]
axes[2].hist(b_confs, bins=10, alpha=0.6, color='#FF6B6B', label='Base Model', range=(0,100))
axes[2].hist(t_confs, bins=10, alpha=0.6, color='#4ECDC4', label='GRPO Trained', range=(0,100))
axes[2].axvline(np.mean(b_confs), color='#FF6B6B', linestyle='--', linewidth=2, label=f'Base avg={np.mean(b_confs):.0f}%')
axes[2].axvline(np.mean(t_confs), color='#4ECDC4', linestyle='--', linewidth=2, label=f'Trained avg={np.mean(t_confs):.0f}%')
axes[2].set_xlabel("Confidence (%)")
axes[2].set_ylabel("Count")
axes[2].set_title("Confidence Distribution")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("echo_baseline_vs_trained.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(f"{'Metric':<25} {'Base Model':>12} {'GRPO Trained':>12} {'Change':>10}")
print("-"*60)
print(f"{'Accuracy':<25} {bm['accuracy']*100:>11.1f}% {tm['accuracy']*100:>11.1f}% {(tm['accuracy']-bm['accuracy'])*100:>+9.1f}%")
print(f"{'ECE (↓ better)':<25} {bm['ece']:>12.4f} {tm['ece']:>12.4f} {(tm['ece']-bm['ece']):>+10.4f}")
print(f"{'Avg Confidence':<25} {bm['avg_confidence']*100:>11.1f}% {tm['avg_confidence']*100:>11.1f}% {(tm['avg_confidence']-bm['avg_confidence'])*100:>+9.1f}%")
print(f"{'Overconfidence Rate':<25} {bm['overconfidence_rate']*100:>11.1f}% {tm['overconfidence_rate']*100:>11.1f}% {(tm['overconfidence_rate']-bm['overconfidence_rate'])*100:>+9.1f}%")
print("="*60)
print("\nPlot saved: echo_baseline_vs_trained.png")
